# TTC Ridership Analysis

This project explores Toronto Transit Commission ridership trends,
seasonal patterns, and recovery compared with 2019.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the data file relative to this notebook, tolerating a kernel started
# from the repo root (one level up) instead of from this folder.
DATA_FILE = Path("Data_source") / "TorontoMeasureData.csv"
data_path = next(
    (p / DATA_FILE for p in (Path.cwd(), Path.cwd() / "TTC") if (p / DATA_FILE).exists()),
    None,
)
if data_path is None:
    raise FileNotFoundError(
        f"{DATA_FILE} not found from {Path.cwd()} - see README 'Verify the data'"
    )

df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")

df.head()

Loaded 508 rows x 4 columns


,Measure Name,Year,Period,Value
0,TTC Average Weekday Ridership,2007,Jan,1400000
1,TTC Average Weekday Ridership,2007,Feb,1486000
2,TTC Average Weekday Ridership,2007,Mar,1485000
3,TTC Average Weekday Ridership,2007,Apr,1491000
4,TTC Average Weekday Ridership,2007,May,1474000


In [2]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.info()

Rows: 508
Columns: 4
<class 'pandas.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Measure Name  508 non-null    str  
 1   Year          508 non-null    int64
 2   Period        462 non-null    str  
 3   Value         508 non-null    int64
dtypes: int64(2), str(2)
memory usage: 16.0 KB


## 🧹 Section 1 — Normalise & shape

Two of the four measures are monthly and two are annual, and they are published on different scales: monthly figures are **raw rides**, annual figures are in **thousands**. This section puts every observation on one schema so the rest of the notebook can join, aggregate and plot without special cases.


### 1.1 Build a tidy analysis frame

The raw export is close to tidy already, but three things would quietly corrupt any
downstream chart if left alone:

1. the two annual measures are published in **thousands**, the monthly ones in **raw rides**;
2. `Period` is null for the annual measures - those 46 nulls are structural, not missing data;
3. `Period` is a plain string, so it sorts alphabetically (Apr, Aug, Dec, ...).

This cell fixes all three and keeps the measurement basis explicit in a `unit` column,
so nothing later can accidentally sum an average into a total.


In [3]:
# --- Section 1.1 - normalise the export into one typed analysis frame ---

MONTH_ORDER = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
MONTH_NUM = {name: i for i, name in enumerate(MONTH_ORDER, start=1)}

WEEKDAY = "TTC Average Weekday Ridership"
MONTHLY = "TTC Monthly Ridership"
PEAK    = "TTC Annual Passenger Rides Peak (000s)"
NONPEAK = "TTC Annual Passenger Rides Non-Peak (000s)"

MONTHLY_MEASURES = [WEEKDAY, MONTHLY]
ANNUAL_MEASURES  = [PEAK, NONPEAK]

# 1) snake_case names; a null Period is exactly what marks an annual observation
clean = (
    df.rename(columns={"Measure Name": "measure", "Year": "year",
                       "Period": "period", "Value": "raw_value"})
      .assign(granularity=lambda d: np.where(d["period"].isna(), "annual", "monthly"))
)

# 2) month number + a real datetime index - monthly observations only
clean["month"] = clean["period"].map(MONTH_NUM).astype("Int8")

clean["date"] = pd.NaT
is_monthly = clean["month"].notna()
clean.loc[is_monthly, "date"] = pd.to_datetime(
    {"year":  clean.loc[is_monthly, "year"],
     "month": clean.loc[is_monthly, "month"].astype("int64"),
     "day":   1}
).to_numpy()

# 3) make the measurement basis explicit: can this number be summed?
clean["unit"] = clean["measure"].map(
    {WEEKDAY: "avg_weekday", MONTHLY: "total", PEAK: "total", NONPEAK: "total"}
)

# 4) annual measures are published in THOUSANDS -> rescale so every value is whole rides
clean["value"] = np.where(
    clean["granularity"] == "annual", clean["raw_value"] * 1000, clean["raw_value"]
).astype("int64")

# 5) flag the incomplete final year, derived from the data rather than hardcoded
PARTIAL_YEAR = int(clean.loc[is_monthly, "year"].max())
clean["is_partial_year"] = clean["year"] == PARTIAL_YEAR

# 6) ordered categoricals so month order is never alphabetical
clean = clean.assign(
    measure=pd.Categorical(clean["measure"],
                           categories=[WEEKDAY, MONTHLY, PEAK, NONPEAK], ordered=True),
    granularity=pd.Categorical(clean["granularity"],
                               categories=["monthly", "annual"], ordered=True),
    unit=pd.Categorical(clean["unit"], categories=["total", "avg_weekday"], ordered=True),
    period=pd.Categorical(clean["period"], categories=MONTH_ORDER, ordered=True),
    year=clean["year"].astype("int16"),
)[["measure", "granularity", "unit", "year",
   "period", "month", "date", "value", "is_partial_year"]]

print(f"clean: {clean.shape[0]:,} rows x {clean.shape[1]} columns")
print(f"       measures     {list(clean['measure'].cat.categories)}")
print(f"       partial year {PARTIAL_YEAR}")
print(f"       dtypes       {', '.join(f'{c}={t}' for c, t in clean.dtypes.astype(str).items())}")
clean.head()

clean: 508 rows x 9 columns
       measures     ['TTC Average Weekday Ridership', 'TTC Monthly Ridership', 'TTC Annual Passenger Rides Peak (000s)', 'TTC Annual Passenger Rides Non-Peak (000s)']
       partial year 2026
       dtypes       measure=category, granularity=category, unit=category, year=int16, period=category, month=Int8, date=datetime64[ns], value=int64, is_partial_year=bool


,measure,granularity,unit,year,period,month,date,value,is_partial_year
0,TTC Average Weekday Ridership,monthly,avg_weekday,2007,Jan,1,2007-01-01,1400000,False
1,TTC Average Weekday Ridership,monthly,avg_weekday,2007,Feb,2,2007-02-01,1486000,False
2,TTC Average Weekday Ridership,monthly,avg_weekday,2007,Mar,3,2007-03-01,1485000,False
3,TTC Average Weekday Ridership,monthly,avg_weekday,2007,Apr,4,2007-04-01,1491000,False
4,TTC Average Weekday Ridership,monthly,avg_weekday,2007,May,5,2007-05-01,1474000,False


### 1.2 Assert the profile (the README, executable)

Every data-quality claim in the README is checked here, so a refreshed upstream file
fails loudly instead of silently changing the story. The reconciliation assert is the
important one: it proves the `x1000` unit conversion is right, because the annual
peak + non-peak totals then reproduce the summed monthly ridership to within rounding.


In [4]:
# --- Section 1.2 - every README data-quality claim, checked in code ---

assert clean.shape == (508, 9), clean.shape
assert clean["value"].notna().all() and (clean["value"] > 0).all(), "null or zero rides"
assert not clean.duplicated(["measure", "year", "month"]).any(), "duplicate keys"

# 237 / 225 monthly, 23 / 23 annual (counts follow the categorical order above)
assert clean["measure"].value_counts(sort=False).tolist() == [237, 225, 23, 23], \
    clean["measure"].value_counts(sort=False).to_dict()
assert clean["granularity"].value_counts().sort_index().to_dict() == \
    {"annual": 46, "monthly": 462}

# the 46 nulls are structural: only the annual measures lack a month
assert clean["period"].isna().sum() == 46
assert clean.loc[clean["period"].isna(), "granularity"].eq("annual").all()
assert clean["month"].isna().sum() == 46 and clean["date"].isna().sum() == 46

# every month 1-12 present, no stray label
assert sorted(clean["month"].dropna().unique().tolist()) == list(range(1, 13))

# the final year is partial and holds Jan-Sep only (9 months x 2 monthly measures)
assert PARTIAL_YEAR == 2026
assert list(clean.loc[clean["is_partial_year"], "period"].unique()) == MONTH_ORDER[:9]
assert clean["is_partial_year"].sum() == 18

# per-measure coverage matches the documented windows
coverage = clean.groupby("measure", observed=True)["year"].agg(["min", "max"])
assert coverage.loc[WEEKDAY].tolist() == [2007, 2026]
assert coverage.loc[MONTHLY].tolist() == [2008, 2026]
assert coverage.loc[PEAK].tolist()    == [2003, 2025]
assert coverage.loc[NONPEAK].tolist() == [2003, 2025]

# --- the x1000 conversion is correct: annual peak+non-peak == summed monthly ---
monthly_sum = clean.query("measure == @MONTHLY").groupby("year")["value"].sum()
annual_sum  = clean.query("measure in @ANNUAL_MEASURES").groupby("year")["value"].sum()
recon = (annual_sum / monthly_sum).dropna()          # only the 2008-2025 overlap
assert (recon - 1).abs().max() <= 2e-4, (recon - 1).abs().max()

# --- headline numbers Sections 2-7 depend on ---
total = clean.query("measure == @MONTHLY").groupby("year")["value"].sum()
assert total[2019] == 525_470_000
assert abs(total[2020] / total[2019] - 1 - (-0.5720)) < 1e-3
assert total.idxmin() == 2021 and total[2021] == 197_841_000
assert abs(total[2025] / total[2019] - 1 - (-0.2141)) < 1e-3

peak  = clean.query("measure == @PEAK").set_index("year")["value"]
nonpk = clean.query("measure == @NONPEAK").set_index("year")["value"]
share = peak / (peak + nonpk)
assert abs(share[2003] - 0.4725) < 1e-4
assert abs(share[2025] - 0.4200) < 1e-4

print(f"validated {len(clean):,} rows | unit reconciliation error "
      f"{(recon - 1).abs().max():.6%} | peak share {share[2003]:.2%} (2003) "
      f"-> {share[2025]:.2%} (2025)")
coverage

validated 508 rows | unit reconciliation error 0.010893% | peak share 47.25% (2003) -> 42.00% (2025)


,min,max
measure,,
TTC Average Weekday Ridership,2007,2026
TTC Monthly Ridership,2008,2026
TTC Annual Passenger Rides Peak (000s),2003,2025
TTC Annual Passenger Rides Non-Peak (000s),2003,2025
